<small>

**PyProjectManagementSim 1.0**  
*Modelo computacional de gestión de proyectos en Python*  

**Autores:** Manuel Castañón–Puga, Fernando Rosales Quiñones  
**Correo:** {puga, fernando.rosales59}@uabc.edu.mx  
**Universidad Autónoma de Baja California**  
[http://www.uabc.edu.mx](http://www.uabc.edu.mx)  
`PyProjectManagementSim/examples/normal_simulations comprehensive_tutorial.ipynb`

</small>

# PyProjectManagementSim - Complete Tutorial

This comprehensive Jupyter notebook demonstrates all the functionalities of the PyProjectManagementSim simulator. You'll learn how to:

1. **Setup and Environment Configuration** - Prepare your environment and verify data files
2. **Data Analysis and Loading** - Analyze XML project files and load real project data
3. **Basic Simulation** - Run simple simulations with different strategies
4. **Visualizing Results** - Plot a run's metrics (completion, CPI, SPI) over time
5. **Advanced Experiments** - Run multiple simulations for statistical analysis
6. **Results Analysis** - Analyze and compare simulation results

## Prerequisites

Make sure you have the following installed:
- Python 3.12+ (required by Mesa 3.5)
- Required packages (see requirements.txt)
- Project data files in the `data/` directory

Let's get started!


## 1. Setup and Environment Configuration

First, let's set up our environment and verify that all required files are available.


In [1]:
import sys
import os
import traceback
from datetime import datetime
from typing import Dict, List, Any, Optional
import json
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import pandas as pd
import time

# Configure matplotlib for better plots
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Required libraries imported successfully!")


Required libraries imported successfully!


In [2]:
# Environment Configuration - Fixed Path Setup
import os
import sys

# Get the current working directory (where the notebook is running)
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")

# Navigate to the project root
# The notebook is in examples/normal_simulations/, so we need to go up two levels
if 'normal_simulations' in current_dir:
    # We're in examples/normal_simulations, go up two levels
    project_root = os.path.normpath(os.path.join(current_dir, '..', '..'))
elif 'examples' in current_dir:
    # We're in examples, go up one level
    project_root = os.path.normpath(os.path.join(current_dir, '..'))
else:
    # We're in the project root
    project_root = current_dir

print(f"Project root: {project_root}")

# Verify the core directory exists
core_path = os.path.join(project_root, 'core')
if os.path.exists(core_path):
    print(f"[OK] Core directory found: {core_path}")
else:
    print(f"[ERROR] Core directory not found: {core_path}")
    print("Available directories:")
    try:
        for item in os.listdir(project_root):
            if os.path.isdir(os.path.join(project_root, item)):
                print(f"  - {item}")
    except:
        print("  Could not list project root directory")

# Verify the data directory exists
data_path = os.path.join(project_root, 'data')
if os.path.exists(data_path):
    print(f"[OK] Data directory found: {data_path}")
    # List data files
    data_files = os.listdir(data_path)
    print(f"  Data files: {data_files}")
else:
    print(f"[ERROR] Data directory not found: {data_path}")

# Add project root to Python path
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"[OK] Added to Python path: {project_root}")

print(f"Python path: {sys.path[:3]}...")  # Show first 3 entries


Current working directory: d:\Universidad\7mo semestre\pre\PyProjectManagementSim\examples\normal_simulations
Project root: d:\Universidad\7mo semestre\pre\PyProjectManagementSim
[OK] Core directory found: d:\Universidad\7mo semestre\pre\PyProjectManagementSim\core
[OK] Data directory found: d:\Universidad\7mo semestre\pre\PyProjectManagementSim\data
  Data files: ['Software Development Plan.mpp', 'Software Development Plan.xml']
[OK] Added to Python path: d:\Universidad\7mo semestre\pre\PyProjectManagementSim
Python path: ['d:\\Universidad\\7mo semestre\\pre\\PyProjectManagementSim', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\\python312.zip', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\\DLLs']...


In [3]:
# Test imports step by step
print("Testing imports...")

try:
    print("1. Testing core module access...")
    import core
    print("[OK] Core module imported successfully")
    
    print("2. Testing core.simulation module...")
    import core.simulation
    print("[OK] core.simulation module imported successfully")
    
    print("3. Testing specific imports...")
    from core.simulation.runner import ExperimentConfig, SimulationRunner
    print("[OK] ExperimentConfig and SimulationRunner imported successfully")
    
    from core.simulation.model import ProjectManagementModel
    print("[OK] ProjectManagementModel imported successfully")
    
    from core.simulation.parameters import SimulationParameters
    print("[OK] SimulationParameters imported successfully")
    
    from core.readers.xml_reader import XmlProjectReader
    print("[OK] XmlProjectReader imported successfully")
    
    from core.project import Project
    print("[OK] Project imported successfully")
    
    print("\n[SUCCESS] All simulator components imported successfully!")
    
except ImportError as e:
    print(f"[ERROR] Import error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running the notebook from the correct directory")
    print("2. Check that the 'core' directory exists in the project root")
    print("3. Verify that all required Python files are present")
    
    # Show current directory structure
    print(f"\nCurrent directory contents:")
    try:
        for item in os.listdir('.'):
            print(f"  - {item}")
    except:
        print("  Could not list current directory")
        
    print(f"\nProject root contents:")
    try:
        for item in os.listdir(project_root):
            print(f"  - {item}")
    except:
        print("  Could not list project root directory")


Testing imports...
1. Testing core module access...
[OK] Core module imported successfully
2. Testing core.simulation module...
Info (AsposeReader): 'aspose-tasks' library imported.
[OK] core.simulation module imported successfully
3. Testing specific imports...
[OK] ExperimentConfig and SimulationRunner imported successfully
[OK] ProjectManagementModel imported successfully
[OK] SimulationParameters imported successfully
[OK] XmlProjectReader imported successfully
[OK] Project imported successfully

[SUCCESS] All simulator components imported successfully!
Info (AsposeReader): 'aspose-tasks' library imported.
[OK] core.simulation module imported successfully
3. Testing specific imports...
[OK] ExperimentConfig and SimulationRunner imported successfully
[OK] ProjectManagementModel imported successfully
[OK] SimulationParameters imported successfully
[OK] XmlProjectReader imported successfully
[OK] Project imported successfully

[SUCCESS] All simulator components imported successfully!


In [4]:
def verify_data_files():
    """Verify that data files exist and show their information."""
    print("=" * 60)
    print("DATA FILE VERIFICATION")
    print("=" * 60)
    
    data_files = [
        os.path.join(project_root, "data", "Software Development Plan.xml"),
        os.path.join(project_root, "data", "Software Development Plan.mpp")
    ]
    
    for data_file in data_files:
        if os.path.exists(data_file):
            print(f"[OK] File found: {os.path.basename(data_file)}")
            file_size = os.path.getsize(data_file)
            print(f"   Size: {file_size:,} bytes")
        else:
            print(f"[WARN] File not found: {os.path.basename(data_file)}")
    
    # Verify output directory
    output_dir = os.path.join(project_root, "core", "generated")
    os.makedirs(output_dir, exist_ok=True)
    print(f"[OK] Output directory prepared: {output_dir}")

# Run verification
verify_data_files()


DATA FILE VERIFICATION
[OK] File found: Software Development Plan.xml
   Size: 395,537 bytes
[OK] File found: Software Development Plan.mpp
   Size: 1,257,472 bytes
[OK] Output directory prepared: d:\Universidad\7mo semestre\pre\PyProjectManagementSim\core\generated


## 2. Data Analysis and Loading

Now let's analyze the XML project file and load the real project data.


In [5]:
def analyze_xml_files():
    """Analyze project XML files and extract real data."""
    print("=" * 60)
    print("XML FILE ANALYSIS")
    print("=" * 60)
    
    xml_file = os.path.join(project_root, "data", "Software Development Plan.xml")
    
    if os.path.exists(xml_file):
        print(f"Analyzing XML file: {os.path.basename(xml_file)}")
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            
            # Define namespace for Microsoft Project XML
            namespace = {'mp': 'http://schemas.microsoft.com/project'}
            
            # Extract real tasks from XML
            tasks_data = []
            for task in root.findall('.//mp:Task', namespace):
                tasks_data.append({
                    "uid": task.find('mp:UID', namespace).text if task.find('mp:UID', namespace) is not None else "N/A",
                    "name": task.find('mp:Name', namespace).text if task.find('mp:Name', namespace) is not None else "Unnamed Task",
                    "duration": task.find('mp:Duration', namespace).text if task.find('mp:Duration', namespace) is not None else "PT0H0M0S",
                    "priority": task.find('mp:Priority', namespace).text if task.find('mp:Priority', namespace) is not None else "500",
                    "start": task.find('mp:Start', namespace).text if task.find('mp:Start', namespace) is not None else "N/A",
                    "finish": task.find('mp:Finish', namespace).text if task.find('mp:Finish', namespace) is not None else "N/A",
                })
            
            # Extract real resources from XML
            resources_data = []
            for resource in root.findall('.//mp:Resource', namespace):
                resources_data.append({
                    "uid": resource.find('mp:UID', namespace).text if resource.find('mp:UID', namespace) is not None else "N/A",
                    "name": resource.find('mp:Name', namespace).text if resource.find('mp:Name', namespace) is not None else "Unnamed Resource",
                    "type": resource.find('mp:Type', namespace).text if resource.find('mp:Type', namespace) is not None else "0",
                    "standard_rate": resource.find('mp:StandardRate', namespace).text if resource.find('mp:StandardRate', namespace) is not None else "0",
                })
            
            # Count elements
            tasks_count = len(tasks_data)
            resources_count = len(resources_data)
            
            print(f"[OK] Analysis completed:")
            print(f"   - Tasks found: {tasks_count}")
            print(f"   - Resources found: {resources_count}")
            
            # Show sample of extracted data
            if tasks_count > 0:
                print(f"   - Sample task: {tasks_data[0]['name']}")
            if resources_count > 0:
                print(f"   - Sample resource: {resources_data[0]['name']}")
            
            return tasks_data, resources_data
            
        except Exception as e:
            print(f"[ERROR] Error analyzing XML: {e}")
            traceback.print_exc()
            return [], []
    else:
        print(f"[WARN] XML file not found: {xml_file}")
        return [], []

# Run XML analysis
xml_tasks, xml_resources = analyze_xml_files()

XML FILE ANALYSIS
Analyzing XML file: Software Development Plan.xml
[OK] Analysis completed:
   - Tasks found: 87
   - Resources found: 9
   - Sample task: Software Development Plan
   - Sample resource: Unnamed Resource


In [6]:
def load_project_data():
    """Load real project data using the XML reader."""
    print("=" * 60)
    print("LOADING REAL PROJECT DATA")
    print("=" * 60)
    
    try:
        # Use the actual XML reader to load project data
        reader = XmlProjectReader()
        xml_path = os.path.join(project_root, "data", "Software Development Plan.xml")
        project_data = reader.load(xml_path)
        
        print(f"[OK] Project data loaded successfully:")
        print(f"   - Tasks loaded: {len(project_data.tasks)}")
        print(f"   - Resources loaded: {len(project_data.resources)}")
        
        # Show sample of loaded data
        if project_data.tasks:
            first_task = list(project_data.tasks.values())[0]
            print(f"   - Sample task: {first_task.name} (Duration: {first_task.duration}h)")
        
        if project_data.resources:
            first_resource = list(project_data.resources.values())[0]
            print(f"   - Sample resource: {first_resource.name} (Cost: ${first_resource.cost_per_hour}/h)")
        
        print(f"[OK] Project data ready for simulation")
        return project_data
        
    except Exception as e:
        print(f"[ERROR] Error loading project data: {e}")
        traceback.print_exc()
        return None

# Load project data
project_data = load_project_data()


LOADING REAL PROJECT DATA
Info (XmlReader): Loading project from 'd:\Universidad\7mo semestre\pre\PyProjectManagementSim\data\Software Development Plan.xml'...
Info (XmlReader): Using namespace: http://schemas.microsoft.com/project
Info (XmlReader): Processing 84 dependencies...
Info (XmlReader): Load completed. Returning ProjectData.
[OK] Project data loaded successfully:
   - Tasks loaded: 74
   - Resources loaded: 8
   - Sample task: Determine project scope (Duration: 4.0h)
   - Sample resource: Management (Cost: $33.0/h)
[OK] Project data ready for simulation


## 3. Basic Simulation

Let's run a basic simulation to understand how the simulator works.


In [7]:
def run_basic_simulation(strategy_name="greedy", num_simulations=3):
    """Run a basic simulation with the specified strategy."""
    print("=" * 60)
    print(f"RUNNING BASIC SIMULATION - {strategy_name.upper()} STRATEGY")
    print("=" * 60)
    
    if project_data is None:
        print("[ERROR] No project data available for simulation")
        return None
    
    # Define team configurations
    team_configs = [
        {
            "id": 101,
            "name": "Analysis and Design Team",
            "member_resource_ids": [1, 2]
        },
        {
            "id": 102,
            "name": "Development and Testing Team", 
            "member_resource_ids": [3, 4]
        }
    ]
    
    try:
        # Create experiment configuration
        config = ExperimentConfig()
        config.mpp_file_name = "Software Development Plan.xml"
        config.data_folder_relative_path = os.path.join(project_root, "data")
        config.reader_type = "xml"
        config.simulation_params = {
            "error_margin": 0.2,
            "reassignment_frequency": 0,
            "max_steps": 2000
        }
        config.team_definitions = team_configs
        config.assignment_strategy = strategy_name
        config.num_simulations = num_simulations
        config.output_filename_base = f"demo_{strategy_name}"
        config.add_timestamp_to_filename = True
        config.output_folder_relative_path = os.path.join(project_root, "core", "generated")
        config.show_loaded_tasks = False
        config.show_loaded_resources = False
        config.display_limit = 0
        config.print_last_run_summary = True
        config.resources_definition = []
        
        # Run simulation
        runner = SimulationRunner(config)
        runner.run()
        
        # Collect results
        if hasattr(runner, 'aggregated_results') and runner.aggregated_results is not None:
            final_results = runner.aggregated_results.iloc[-1]
            
            simulation_result = {
                "strategy": strategy_name,
                "simulations_run": num_simulations,
                "final_time": final_results.get('Time', 0),
                "completion_percentage": final_results.get('Completion', 0),
                "actual_cost": final_results.get('ActualCost', 0),
                "earned_value": final_results.get('EarnedValue', 0),
                "cpi": final_results.get('CPI', 0),
                "spi": final_results.get('SPI', 0),
                "active_agents": final_results.get('ActiveAgents', 0),
                "avg_team_morale": final_results.get('AvgTeamMorale', 0),
                "avg_team_cohesion": final_results.get('AvgTeamCohesion', 0),
                "timestamp": datetime.now().isoformat()
            }
            
            print(f"[OK] Strategy: {strategy_name}")
            print(f"[OK] Simulations completed: {num_simulations}")
            print(f"[OK] Final completion: {simulation_result['completion_percentage']:.2f}%")
            print(f"[OK] Final cost: ${simulation_result['actual_cost']:,.2f}")
            print(f"[OK] CPI: {simulation_result['cpi']:.3f}")
            print(f"[OK] SPI: {simulation_result['spi']:.3f}")
            print(f"[OK] Team morale: {simulation_result['avg_team_morale']:.3f}")
            
            return simulation_result
            
        else:
            print(f"[WARN]  No results available for strategy: {strategy_name}")
            return None
            
    except Exception as e:
        print(f"[ERROR] Error running simulation for {strategy_name}: {e}")
        traceback.print_exc()
        return None

# Run basic simulation with greedy strategy
basic_result = run_basic_simulation("greedy", 3)


RUNNING BASIC SIMULATION - GREEDY STRATEGY

--- Starting 3 Simulations ---
Info (Runner): Preparing project from 'd:\Universidad\7mo semestre\pre\PyProjectManagementSim\data\Software Development Plan.xml'...
Info (XmlReader): Loading project from 'd:\Universidad\7mo semestre\pre\PyProjectManagementSim\data\Software Development Plan.xml'...
Info (XmlReader): Using namespace: http://schemas.microsoft.com/project
Info (XmlReader): Processing 84 dependencies...
Info (XmlReader): Load completed. Returning ProjectData.
Info (Project): Dependencies resolved for 74 tasks.
Info (Runner): Simulation parameters ready: SimulationParameters(error_margin=0.2, reassignment_frequency=0, max_steps=2000, team_definitions=[{'id': 101, 'name': 'Analysis and Design Team', 'member_resource_ids': [1, 2]}, {'id': 102, 'name': 'Development and Testing Team', 'member_resource_ids': [3, 4]}], assignment_strategy='greedy')
--- Simulation 1/3 ---

Creating teams from configuration...
 -> Team Created: Team 101: An

## 4. Visualizing Results (optional)

> Overview
- The simulator runs in batch mode; there is no live view. Each run's metrics are saved per step in the CSV files written to `core/generated/`.
- Columns include `Time`, `Completion`, `ActualCost`, `EarnedValue`, `CPI`, `SPI` and `simulation_id`.

> Example: plot CPI and SPI of the latest basic simulation (run Step 3 first)
```python
import glob
import pandas as pd
import matplotlib.pyplot as plt

latest = max(glob.glob(os.path.join(project_root, "core", "generated", "demo_*.csv")), key=os.path.getmtime)
df = pd.read_csv(latest)
df[df["simulation_id"] == 1].plot(x="Time", y=["CPI", "SPI"], title=os.path.basename(latest))
plt.show()
```

> Next steps
- Run a basic simulation in Step 3 (above) or jump to Step 5 to run multiple strategies.

## 5. Advanced Experiments and Results

> Run multiple strategies
- Re-run the basic simulation function with different strategies:
  - `basic_result = run_basic_simulation("greedy", 5)`
  - `basic_result = run_basic_simulation("shortest_first", 5)`

> Analyze outputs
- CSV results are written to `core/generated/` with a timestamped filename.
- Load them with pandas (see the example in Step 4) to compare runs, or open them in any spreadsheet tool.

> Tip
- To ensure a clean environment, consider restarting the kernel and running from the top before long experiments.

## 6. Quick Comparison Runner (optional)

> This cell will run two strategies back-to-back (greedy and shortest_first), display a compact summary, and save a CSV with the comparison into `core/generated/`.

In [9]:
# Run two strategies and compare results
try:
    # Ensure project_data and run_basic_simulation are available
    if 'project_data' not in globals() or project_data is None:
        print("[INFO] Reloading project data...")
        project_data = load_project_data()
    
    print("\n[STEP] Running quick comparison: greedy vs shortest_first")
    res_greedy = run_basic_simulation("greedy", 5)
    res_shortest = run_basic_simulation("shortest_first", 5)
    
    results = []
    for r in [res_greedy, res_shortest]:
        if r is not None:
            results.append({
                "strategy": r.get("strategy"),
                "simulations_run": r.get("simulations_run"),
                "completion_%": round(float(r.get("completion_percentage", 0)), 2),
                "actual_cost": float(r.get("actual_cost", 0)),
                "CPI": round(float(r.get("cpi", 0)), 3),
                "SPI": round(float(r.get("spi", 0)), 3),
                "avg_morale": round(float(r.get("avg_team_morale", 0)), 3),
                "avg_cohesion": round(float(r.get("avg_team_cohesion", 0)), 3),
            })
    
    if results:
        df = pd.DataFrame(results)
        display(df) if 'display' in globals() else print(df)
        
        # Save to CSV
        out_dir = os.path.join(project_root, "core", "generated")
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, f"quick_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        df.to_csv(out_path, index=False)
        print(f"[OK] Comparison saved to: {out_path}")
    else:
        print("[WARN] No results to compare.")
except Exception as e:
    print(f"[ERROR] Quick comparison failed: {e}")
    traceback.print_exc()


[STEP] Running quick comparison: greedy vs shortest_first
RUNNING BASIC SIMULATION - GREEDY STRATEGY

--- Starting 5 Simulations ---
Info (Runner): Preparing project from 'd:\Universidad\7mo semestre\pre\PyProjectManagementSim\data\Software Development Plan.xml'...
Info (XmlReader): Loading project from 'd:\Universidad\7mo semestre\pre\PyProjectManagementSim\data\Software Development Plan.xml'...
Info (XmlReader): Using namespace: http://schemas.microsoft.com/project
Info (XmlReader): Processing 84 dependencies...
Info (XmlReader): Load completed. Returning ProjectData.
Info (Project): Dependencies resolved for 74 tasks.
Info (Runner): Simulation parameters ready: SimulationParameters(error_margin=0.2, reassignment_frequency=0, max_steps=2000, team_definitions=[{'id': 101, 'name': 'Analysis and Design Team', 'member_resource_ids': [1, 2]}, {'id': 102, 'name': 'Development and Testing Team', 'member_resource_ids': [3, 4]}], assignment_strategy='greedy')
--- Simulation 1/5 ---

Creating